# 0. Verificación del Entorno

**Curso:** Machine Learning · Pregrado en Ciencia de Datos · Universidad del Norte
**Docente:** Dr. Lihki Rubio
**Equipo:** Juan Camilo Conrado · Sergio Cadavid · Mateo Chang

---

Este notebook valida que el entorno está correctamente configurado **antes** de proceder con los demás. Verifica:

1. Versión de Python correcta (3.10+).
2. Todas las librerías importables.
3. Dataset en su lugar.
4. Módulos del proyecto (`src/`) accesibles.
5. Tests anti-leakage pasando.

> Si alguno de los chequeos falla, detén la ejecución y arregla el problema antes de avanzar al notebook 01.


In [1]:
# Asegurar acceso al paquete src/ desde notebooks/
import sys
from pathlib import Path
_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

## 1. Versión de Python

In [2]:
import sys
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 10), "Se requiere Python >= 3.10"
print("✅ Versión de Python correcta")

Python: 3.11.15 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:12:15) [MSC v.1942 64 bit (AMD64)]
✅ Versión de Python correcta


## 2. Librerías obligatorias

In [3]:
# Core
import numpy as np
import pandas as pd
import matplotlib
import seaborn as sns
import scipy

# ML
import sklearn
import xgboost
import imblearn

# Series de tiempo
import statsmodels

# Storage
import pyarrow
import joblib

versions = {
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "seaborn": sns.__version__,
    "scipy": scipy.__version__,
    "scikit-learn": sklearn.__version__,
    "xgboost": xgboost.__version__,
    "imbalanced-learn": imblearn.__version__,
    "statsmodels": statsmodels.__version__,
    "pyarrow": pyarrow.__version__,
    "joblib": joblib.__version__,
}

for lib, v in versions.items():
    print(f"  {lib:25s} {v}")

print("\n✅ Librerías core importadas correctamente")

  numpy                     1.26.4
  pandas                    2.2.2
  matplotlib                3.8.4
  seaborn                   0.13.2
  scipy                     1.13.0
  scikit-learn              1.4.2
  xgboost                   2.0.3
  imbalanced-learn          0.12.2
  statsmodels               0.14.2
  pyarrow                   16.1.0
  joblib                    1.4.2

✅ Librerías core importadas correctamente


## 3. Librerías opcionales (avanzadas)

In [4]:
optional_libs = {}

try:
    import optuna
    optional_libs["optuna"] = optuna.__version__
except ImportError:
    optional_libs["optuna"] = "❌ NO INSTALADA"

try:
    import deap
    optional_libs["deap"] = deap.__version__
except ImportError:
    optional_libs["deap"] = "❌ NO INSTALADA"

try:
    import lime
    optional_libs["lime"] = lime.__version__
except ImportError:
    optional_libs["lime"] = "❌ NO INSTALADA"

try:
    import arch
    optional_libs["arch (GARCH)"] = arch.__version__
except ImportError:
    optional_libs["arch (GARCH)"] = "❌ NO INSTALADA — instalar con 'pip install arch'"

try:
    import lightgbm
    optional_libs["lightgbm"] = lightgbm.__version__
except ImportError:
    optional_libs["lightgbm"] = "⚠️ no disponible (notebook 11 lo reportará)"

try:
    import catboost
    optional_libs["catboost"] = catboost.__version__
except ImportError:
    optional_libs["catboost"] = "⚠️ no disponible (notebook 11 lo reportará)"

try:
    import tensorflow as tf
    optional_libs["tensorflow"] = tf.__version__
except ImportError:
    optional_libs["tensorflow"] = "⚠️ no disponible (LSTM no se podrá entrenar)"

try:
    import pmdarima
    optional_libs["pmdarima"] = pmdarima.__version__
except ImportError:
    optional_libs["pmdarima"] = "⚠️ no disponible (se usará statsmodels.ARIMA directo)"

for lib, v in optional_libs.items():
    print(f"  {lib:25s} {v}")

AttributeError: module 'lime' has no attribute '__version__'

---

> **📊 Interpretación:** Las librerías marcadas con ⚠️ son opcionales: si no se instalan, los notebooks correspondientes (especialmente el 11 de modelos avanzados) reportan la ausencia como limitación pero el proyecto sigue siendo reproducible. Las librerías marcadas con ❌ son obligatorias y deben instalarse antes de continuar.


## 4. Módulos del proyecto

In [ ]:
# Importar todos los módulos del proyecto src/
from src.config import (RANDOM_STATE, ROOT_DIR, DATASET_FILE,
                        TRAIN_FRAC, VAL_FRAC, TEST_FRAC,
                        HORIZONS, VOL_WINDOWS)
from src.data_loader import load_intc, quick_qa
from src.features import build_features, get_feature_columns
from src.targets import build_all_targets, build_regime_target
from src.splits import temporal_split, make_tscv, split_summary
from src.pipelines import (get_regression_pipelines,
                            get_classification_pipelines,
                            make_imb_pipeline)
from src.benchmarks import (NaiveForecast, RollingMeanForecast,
                             EWMAForecast, ARIMAForecast, GARCHForecast)
from src.stats_tests import (diebold_mariano, delong_test,
                              bootstrap_metric, bonferroni_correction,
                              full_residual_diagnostics)
from src.interpretability import (xgb_feature_importance,
                                   lime_explainer, explain_with_lime)
from src.hvrf import HVRF
from src.viz import set_style, color_for, save_fig
from src.io_utils import (save_model, load_model,
                           save_predictions_df, load_predictions_df,
                           save_metrics, load_metrics,
                           save_processed, load_processed)

print("✅ Todos los módulos del proyecto src/ importados correctamente")
print(f"\nROOT_DIR: {ROOT_DIR}")
print(f"DATASET_FILE: {DATASET_FILE}")
print(f"RANDOM_STATE: {RANDOM_STATE}")

## 5. Dataset disponible

In [ ]:
assert DATASET_FILE.exists(), f"Dataset no encontrado en {DATASET_FILE}"

df_raw = load_intc(filter_modern=False)
df = load_intc(filter_modern=True)

print(f"Dataset completo:        {len(df_raw):>6} filas")
print(f"Dataset filtrado (≥1990): {len(df):>6} filas")
print(f"Período usado:           {df['date'].min().date()} → {df['date'].max().date()}")

qa = quick_qa(df)
print(f"\nQA del dataset filtrado:")
print(f"  Días hábiles esperados:  {qa['n_business_days_expected']}")
print(f"  Días presentes:          {qa['n_business_days_present']}")
print(f"  Días faltantes:          {qa['n_business_days_missing']} ({qa['pct_business_days_missing']:.2f}%)")
print(f"  Duplicados de fecha:     {qa['n_duplicates_date']}")
print(f"  Errores OHLC:            {qa['n_high_lt_low'] + qa['n_high_lt_open_close'] + qa['n_low_gt_open_close']}")

---

> **📊 Interpretación:** El dataset cubre el período completo deseado (1990–2017) con baja tasa de días faltantes y sin errores en la consistencia de OHLC. La diferencia entre filas crudas y filtradas (~4,500 filas) corresponde al período pre-1990 que se descarta por precios artefactuales tras los splits accionarios de Intel.


## 6. Tests anti-leakage

In [ ]:
import subprocess
result = subprocess.run(
    ["python", "-m", "pytest", "tests/", "-v", "--tb=short"],
    capture_output=True,
    text=True,
    cwd=str(ROOT_DIR),
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("❌ Algún test falló. Revisar antes de continuar.")
print("✅ Todos los tests pasan")

---

> **📊 Interpretación:** Los 8 tests verifican (1) que ninguna feature en fecha t use información de fechas posteriores a t, (2) que no queden infinitos tras el feature engineering, (3) que el conteo de features esté en el rango esperado, (4) que el split temporal sea estrictamente cronológico, (5) que las proporciones train/val/test sean razonables, (6) que `target_vol_7[t] == vol_7[t+7]` exactamente, (7) que el target binario sea efectivamente binario, y (8) que el balance de clases sea razonable. Si los 8 tests pasan, la base anti-leakage del proyecto está garantizada.


## 7. Resumen del entorno

Si llegaste hasta aquí sin errores, el entorno está listo. Procede al notebook **`01_eda.ipynb`** para iniciar el análisis exploratorio.

| Verificación | Estado |
|---|---|
| Python 3.10+ | ✅ |
| Librerías core | ✅ |
| Módulos del proyecto | ✅ |
| Dataset disponible | ✅ |
| Tests anti-leakage | ✅ |

> **Nota práctica:** este notebook se ejecuta en menos de 30 segundos. Si tarda más, probablemente alguna de las librerías opcionales está descargando wheels. Es seguro continuar.
